In [1]:
# Reading the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

/Users/allaaboudouara/Desktop/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
transactions = pd.read_csv("transaction_data.csv")
products = pd.read_csv("product.csv")

In [3]:
transactions.head()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [4]:
transactions.shape

(2595732, 12)

In [5]:
products.head()

,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


In [6]:
products.shape

(92353, 7)

### reduce size of dataset

##### Choose 10K unique baskets

In [7]:
sample_baskets = transactions['BASKET_ID'].unique()[:10000]

transactions_small = transactions[
    transactions['BASKET_ID'].isin(sample_baskets)
]

##### remove unpopular products

In [8]:
product_counts = transactions_small['PRODUCT_ID'].value_counts()

popular_products = product_counts[product_counts >= 5].index

transactions_small = transactions_small[
    transactions_small['PRODUCT_ID'].isin(popular_products)
]

##### remove empty baskets

In [9]:
merged_df = transactions_small.merge(products, on='PRODUCT_ID')

In [10]:
basket_counts = merged_df.groupby('BASKET_ID')['PRODUCT_ID'].count()

valid_baskets = basket_counts[
    basket_counts > 1
].index

merged_df = merged_df[
    merged_df['BASKET_ID'].isin(valid_baskets)
]

In [11]:
merged_df = merged_df[
    [
        'household_key',
        'BASKET_ID',
        'PRODUCT_ID',
        'QUANTITY',
        'SALES_VALUE',
        'DEPARTMENT',
        'BRAND',
        'COMMODITY_DESC',
        'SUB_COMMODITY_DESC'
    ]
]

merged_df.head()

,household_key,BASKET_ID,PRODUCT_ID,QUANTITY,SALES_VALUE,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC
0,2375,26984851472,1004906,1,1.39,PRODUCE,Private,POTATOES,POTATOES RUSSET (BULK&BAG)
1,2375,26984851472,1033142,1,0.82,PRODUCE,National,ONIONS,ONIONS SWEET (BULK&BAG)
2,2375,26984851472,1036325,1,0.99,PRODUCE,Private,VEGETABLES - ALL OTHERS,CELERY
3,2375,26984851472,1082185,1,1.21,PRODUCE,National,TROPICAL FRUIT,BANANAS
4,2375,26984851472,8160430,1,1.50,PRODUCE,Private,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS


In [12]:
merged_df.shape

(62250, 9)

# Creation of new features for content and item based filtering systems

### Product text

In [13]:
merged_df['ProductText'] = (
    merged_df['COMMODITY_DESC'].fillna('') + ' ' +
    merged_df['SUB_COMMODITY_DESC'].fillna('') + ' ' +
    merged_df['BRAND'].fillna('') + ' ' +
    merged_df['DEPARTMENT'].fillna('')
)

### Average Basket Size per household

In [14]:
basket_products = (
    merged_df.groupby('BASKET_ID')['PRODUCT_ID']
    .nunique()
    .reset_index(name='UniqueProducts')
)

basket_products = basket_products.merge(
    merged_df[['BASKET_ID', 'household_key']].drop_duplicates(),
    on='BASKET_ID'
)

average_basket_size = (
    basket_products.groupby('household_key')['UniqueProducts']
    .mean()
    .reset_index(name='AverageBasketSize')
)

merged_df = merged_df.merge(
    average_basket_size,
    on='household_key',
    how='left'
)

### Household spending

In [15]:
household_spending = (
    merged_df.groupby('household_key')['SALES_VALUE']
    .sum()
    .reset_index(name='HouseholdSpending')
)

merged_df = merged_df.merge(
    household_spending,
    on='household_key',
    how='left'
)

### Household Purchase Frequency

In [16]:
purchase_frequency = (
    merged_df.groupby('household_key')['BASKET_ID']
    .nunique()
    .reset_index(name='PurchaseFrequency')
)

merged_df = merged_df.merge(
    purchase_frequency,
    on='household_key',
    how='left'
)

### Product Popularity

In [17]:
product_popularity = (
    merged_df.groupby('PRODUCT_ID')['household_key']
    .count()
    .reset_index(name='ProductPopularity')
)

merged_df = merged_df.merge(
    product_popularity,
    on='PRODUCT_ID',
    how='left'
)

### Interaction Score

In [18]:
merged_df['InteractionScore'] = (
    merged_df['QUANTITY']
    * np.log1p(merged_df['HouseholdSpending'])
    * np.log1p(merged_df['PurchaseFrequency'])
)

In [19]:
# Question 1

# Content Based filtering recommendation System

In [20]:
products = (
    merged_df[['PRODUCT_ID', 'ProductText']]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(products['ProductText'])

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [23]:
indices = pd.Series(
    products.index,
    index=products['PRODUCT_ID']
).drop_duplicates()

In [24]:
def recommend_products(product_id, cosine_sim=cosine_sim):

    if product_id not in indices.index:
        print("Product ID not found.")
        return
        
    idx = indices[product_id]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:6]

    product_indices = [i[0] for i in sim_scores]

    return products.iloc[product_indices]

In [25]:
products['PRODUCT_ID'].unique()[:20]

array([1004906, 1033142, 1036325, 1082185, 8160430,  826249, 1043142,
       1085983, 1102651,  842930,  981760,  833715,  866950, 1022843,
       1048462, 1071333,  923972, 1131351,  878302,  965138])

In [26]:
recommend_products(842930)

,PRODUCT_ID,ProductText
1396,6979128,CONVENIENT BRKFST/WHLSM SNACKS GRANOLA BARS Pr...
2524,909750,CONVENIENT BRKFST/WHLSM SNACKS GRANOLA BARS Pr...
3026,856335,CONVENIENT BRKFST/WHLSM SNACKS GRANOLA BARS Pr...
3287,6979160,CONVENIENT BRKFST/WHLSM SNACKS GRANOLA BARS Pr...
3779,819330,CONVENIENT BRKFST/WHLSM SNACKS GRANOLA BARS Pr...


#### Same recommder but better formating to see the orginal and recommended products

In [27]:
product_lookup = (
    merged_df[
        [
            'PRODUCT_ID',
            'COMMODITY_DESC',
            'SUB_COMMODITY_DESC',
            'BRAND'
        ]
    ]
    .drop_duplicates()
)

In [28]:
def recommend_products(product_id, cosine_sim=cosine_sim):

    if product_id not in indices.index:
        print("Product ID not found.")
        return

    idx = indices[product_id]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:6]

    product_indices = [i[0] for i in sim_scores]

    recommended_products = products.iloc[product_indices]

    # Get original product info
    original_product = product_lookup[
        product_lookup['PRODUCT_ID'] == product_id
    ]

    print("ORIGINAL PRODUCT\n")
    print(original_product)

    print("\nRECOMMENDED PRODUCTS")

    return product_lookup[
        product_lookup['PRODUCT_ID']
        .isin(recommended_products['PRODUCT_ID'])
    ]

In [29]:
recommend_products(842930)

ORIGINAL PRODUCT

   PRODUCT_ID                  COMMODITY_DESC SUB_COMMODITY_DESC    BRAND
9      842930  CONVENIENT BRKFST/WHLSM SNACKS       GRANOLA BARS  Private

RECOMMENDED PRODUCTS


,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND
2119,6979128,CONVENIENT BRKFST/WHLSM SNACKS,GRANOLA BARS,Private
5186,909750,CONVENIENT BRKFST/WHLSM SNACKS,GRANOLA BARS,Private
7321,856335,CONVENIENT BRKFST/WHLSM SNACKS,GRANOLA BARS,Private
8639,6979160,CONVENIENT BRKFST/WHLSM SNACKS,GRANOLA BARS,Private
12361,819330,CONVENIENT BRKFST/WHLSM SNACKS,GRANOLA BARS,Private


In [30]:
recommend_products(1022843)

ORIGINAL PRODUCT

    PRODUCT_ID COMMODITY_DESC        SUB_COMMODITY_DESC     BRAND
13     1022843           SOUP  RAMEN NOODLES/RAMEN CUPS  National

RECOMMENDED PRODUCTS


,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND
12,866950,SOUP,RAMEN NOODLES/RAMEN CUPS,National
13,1022843,SOUP,RAMEN NOODLES/RAMEN CUPS,National
15,1071333,SOUP,RAMEN NOODLES/RAMEN CUPS,National
116,979585,SOUP,RAMEN NOODLES/RAMEN CUPS,National
183,981944,SOUP,RAMEN NOODLES/RAMEN CUPS,National


In [31]:
product_lookup = (
    merged_df[
        [
            'PRODUCT_ID',
            'COMMODITY_DESC',
            'SUB_COMMODITY_DESC',
            'BRAND'
        ]
    ]
    .drop_duplicates()
)

In [32]:
def recommend_products(product_id, cosine_sim=cosine_sim):

    if product_id not in indices.index:
        print("Product ID not found.")
        return

    idx = indices[product_id]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:6]

    product_indices = [i[0] for i in sim_scores]

    recommended_products = products.iloc[product_indices]

    # Get original product info
    original_product = product_lookup[
        product_lookup['PRODUCT_ID'] == product_id
    ]

    print("ORIGINAL PRODUCT\n")
    print(original_product)

    print("\nRECOMMENDED PRODUCTS")

    return product_lookup[
        product_lookup['PRODUCT_ID']
        .isin(recommended_products['PRODUCT_ID'])
    ]

In [33]:
recommend_products(1022843)

ORIGINAL PRODUCT

    PRODUCT_ID COMMODITY_DESC        SUB_COMMODITY_DESC     BRAND
13     1022843           SOUP  RAMEN NOODLES/RAMEN CUPS  National

RECOMMENDED PRODUCTS


,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND
12,866950,SOUP,RAMEN NOODLES/RAMEN CUPS,National
13,1022843,SOUP,RAMEN NOODLES/RAMEN CUPS,National
15,1071333,SOUP,RAMEN NOODLES/RAMEN CUPS,National
116,979585,SOUP,RAMEN NOODLES/RAMEN CUPS,National
183,981944,SOUP,RAMEN NOODLES/RAMEN CUPS,National


# Item-Item Based Recommendation system

In [34]:
user_item_matrix = merged_df.pivot_table(
    index='household_key',
    columns='PRODUCT_ID',
    values='InteractionScore',
    aggfunc='sum',
    fill_value=0
)

In [35]:
user_item_matrix = merged_df.pivot_table(
    index='household_key',
    columns='PRODUCT_ID',
    values='InteractionScore',
    aggfunc='sum',
    fill_value=0
)

In [36]:
from scipy.sparse import csr_matrix

matrix = csr_matrix(user_item_matrix.values)

In [37]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

model.fit(matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [38]:
item_matrix = user_item_matrix.T

In [39]:
item_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

item_model.fit(item_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [40]:
product_lookup = (
    merged_df[
        [
            'PRODUCT_ID',
            'COMMODITY_DESC',
            'SUB_COMMODITY_DESC',
            'BRAND',
            'DEPARTMENT'
        ]
    ]
    .drop_duplicates()
)

In [41]:
def recommend_item(product_id, n_recommendations=5):

    # Check if product exists
    if product_id not in item_matrix.index:
        print("Product ID not found.")
        return

    # Find nearest items
    distances, indices = item_model.kneighbors(
        item_matrix.loc[product_id].values.reshape(1, -1),
        n_neighbors=n_recommendations + 1
    )

    recommendations = []

    for i in range(1, len(distances.flatten())):

        recommended_product = item_matrix.index[
            indices.flatten()[i]
        ]

        # Lookup metadata
        product_info = product_lookup[
            product_lookup['PRODUCT_ID'] == recommended_product
        ].iloc[0]

        similarity_score = 1 - distances.flatten()[i]

        recommendations.append({
            'PRODUCT_ID': recommended_product,
            'COMMODITY_DESC': product_info['COMMODITY_DESC'],
            'SUB_COMMODITY_DESC': product_info['SUB_COMMODITY_DESC'],
            'BRAND': product_info['BRAND'],
            'DEPARTMENT': product_info['DEPARTMENT'],
            'SimilarityScore': round(similarity_score, 3)
        })

    # Show original product
    original_product = product_lookup[
        product_lookup['PRODUCT_ID'] == product_id
    ]

    print("ORIGINAL PRODUCT\n")
    print(pd.DataFrame(original_product))

    print("\nRECOMMENDED PRODUCTS")

    return pd.DataFrame(recommendations)

In [42]:
recommend_item(842930)

ORIGINAL PRODUCT

   PRODUCT_ID                  COMMODITY_DESC SUB_COMMODITY_DESC    BRAND  \
9      842930  CONVENIENT BRKFST/WHLSM SNACKS       GRANOLA BARS  Private   

  DEPARTMENT  
9    GROCERY  

RECOMMENDED PRODUCTS


,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT,SimilarityScore
0,1071759,CONVENIENT BRKFST/WHLSM SNACKS,GRANOLA BARS,Private,GROCERY,0.829
1,852963,COLD CEREAL,ADULT CEREAL,National,GROCERY,0.763
2,6773238,FROZEN PIZZA,PIZZA/ECONOMY,Private,GROCERY,0.671
3,1071594,FROZEN PIZZA,PIZZA/PREMIUM,National,GROCERY,0.548
4,6703856,HOUSEHOLD CLEANG NEEDS,TOOLS - BATHROOM & TOILET,National,GROCERY,0.517


In [43]:
recommend_item(965138)

ORIGINAL PRODUCT

    PRODUCT_ID COMMODITY_DESC SUB_COMMODITY_DESC    BRAND DEPARTMENT
19      965138  COOKIES/CONES   SANDWICH COOKIES  Private    GROCERY

RECOMMENDED PRODUCTS


,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT,SimilarityScore
0,857315,BEEF,ROUND/LEG - STK/CHP/SLC,National,MEAT,0.632
1,1028816,TROPICAL FRUIT,AVOCADO,National,PRODUCE,0.591
2,9935616,SOFT DRINKS,SOFT DRINKS 12/18&15PK CAN CAR,National,GROCERY,0.492
3,1040848,BAG SNACKS,TORTILLA/NACHO CHIPS,Private,GROCERY,0.489
4,1135044,FRZN FRUITS,FROZEN FRUIT,Private,GROCERY,0.480


In [44]:
recommend_item(1048462)

ORIGINAL PRODUCT

    PRODUCT_ID          COMMODITY_DESC      SUB_COMMODITY_DESC     BRAND  \
14     1048462  BAKED BREAD/BUNS/ROLLS  MAINSTREAM WHITE BREAD  National   

   DEPARTMENT  
14    GROCERY  

RECOMMENDED PRODUCTS


,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT,SimilarityScore
0,915349,YOGURT,YOGURT NOT MULTI-PACKS,National,GROCERY,0.622
1,1066796,CANNED JUICES,ASEPTIC PACK JUICE AND DRINKS,National,GROCERY,0.616
2,986293,CANNED JUICES,ASEPTIC PACK JUICE AND DRINKS,National,GROCERY,0.615
3,1038985,REFRGRATD JUICES/DRNKS,DAIRY CASE 100% PURE JUICE - O,Private,GROCERY,0.606
4,946273,MOLASSES/SYRUP/PANCAKE MIXS,PANCAKE MIXES,Private,GROCERY,0.588


In [45]:
recommend_item(1082185)

ORIGINAL PRODUCT

   PRODUCT_ID  COMMODITY_DESC SUB_COMMODITY_DESC     BRAND DEPARTMENT
3     1082185  TROPICAL FRUIT            BANANAS  National    PRODUCE

RECOMMENDED PRODUCTS


,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT,SimilarityScore
0,1127831,BERRIES,STRAWBERRIES,National,PRODUCE,0.591
1,854852,TOMATOES,TOMATOES HOTHOUSE ON THE VINE,National,PRODUCE,0.579
2,986947,CITRUS,ORANGES NAVELS ALL,National,PRODUCE,0.474
3,878996,GRAPES,GRAPES RED,National,PRODUCE,0.455
4,961554,CARROTS,CARROTS MINI PEELED,Private,PRODUCE,0.454


In [46]:
recommend_item(1071333)

ORIGINAL PRODUCT

    PRODUCT_ID COMMODITY_DESC        SUB_COMMODITY_DESC     BRAND DEPARTMENT
15     1071333           SOUP  RAMEN NOODLES/RAMEN CUPS  National    GROCERY

RECOMMENDED PRODUCTS


,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT,SimilarityScore
0,960999,REFRGRATD JUICES/DRNKS,DAIRY CASE 100% PURE JUICE OTH,Private,GROCERY,0.769
1,959179,COLD CEREAL,ALL FAMILY CEREAL,National,GROCERY,0.764
2,6464094,CANNED JUICES,ASEPTIC PACK JUICE AND DRINKS,Private,GROCERY,0.720
3,846833,FD WRAPS/BAGS/TRSH BG,GARBAGE KITCHEN COMPACTOR BA,Private,GROCERY,0.685
4,883428,SOFT DRINKS,CARBONATED WATER - UNFLAVORED,Private,GROCERY,0.560


# User-User Based Recommendation system

In [47]:
from sklearn.neighbors import NearestNeighbors

# User-user model
user_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

user_model.fit(user_item_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [48]:
def recommend_for_user(household_id, n_recommendations=5):

    if household_id not in user_item_matrix.index:
        print("Household not found.")
        return

    distances, indices = user_model.kneighbors(
        user_item_matrix.loc[household_id].values.reshape(1, -1),
        n_neighbors=6
    )

    similar_users = indices.flatten()[1:]

    similar_user_ids = user_item_matrix.index[similar_users]

    # Products already purchased
    user_products = set(
        merged_df[
            merged_df['household_key'] == household_id
        ]['PRODUCT_ID']
    )

    recommendations = []

    for sim_user in similar_user_ids:

        sim_user_products = merged_df[
            merged_df['household_key'] == sim_user
        ]['PRODUCT_ID']

        for product in sim_user_products:

            if product not in user_products:
                recommendations.append(product)

    recommendations = list(set(recommendations))[:n_recommendations]

    return product_lookup[
        product_lookup['PRODUCT_ID'].isin(recommendations)
    ]

In [49]:
recommend_for_user(2375)

,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT
330,825994,VALUE ADDED FRUIT,INSTORE CUT FRUIT,National,PRODUCE
1284,1005186,SALAD BAR,SALAD BAR FRESH FRUIT,National,SALAD BAR
1617,951190,SALD DRSNG/SNDWCH SPRD,POURABLE SALAD DRESSINGS,National,GROCERY
7501,5564303,POTATOES,POTATOES OTHER,National,PRODUCE
23686,6391173,JUICE,JUICE,National,NUTRITION


In [50]:
recommend_for_user(1130)

,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT
10,981760,EGGS,EGGS - X-LARGE,Private,GROCERY
517,943233,BAKED SWEET GOODS,SNACK CAKE - MULTI PACK,National,GROCERY
2126,852360,REFRGRATD JUICES/DRNKS,DAIRY CASE 100% PURE JUICE - O,Private,GROCERY
6299,1068292,CHICKEN,CHICKEN DRUMS,National,MEAT
24137,1091976,CONDIMENTS/SAUCES,SALAD MUSTARD,Private,GROCERY


In [51]:
recommend_for_user(1617)

,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT
364,833025,MILK BY-PRODUCTS,SOUR CREAMS,Private,GROCERY
685,1042438,COOKIES/CONES,SANDWICH COOKIES,National,GROCERY
1075,827919,FLOUR & MEALS,FLOUR: WHITE & SELF RISING,National,GROCERY
2879,1095700,DINNER SAUSAGE,SMOKED/COOKED,National,MEAT-PCKGD
9833,1013770,FRZN VEGETABLE/VEG DSH,FRZN BAGGED VEGETABLES - PLAIN,Private,GROCERY


In [52]:
recommend_for_user(1060)

,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT
176,7441419,ICE CREAM/MILK/SHERBTS,PREMIUM,Private,GROCERY
961,951821,LUNCHMEAT,MISCELLANEOUS,National,MEAT-PCKGD
1136,1007116,FROZEN PIE/DESSERTS,FRZN WHIPPED TOPPING,Private,GROCERY
3563,7467024,PIES,PIES: FRUIT/NUT,National,PASTRY
7130,871433,YOGURT,YOGURT NOT MULTI-PACKS,National,GROCERY


In [53]:
recommend_for_user(1172)

,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT
596,1138189,SOFT DRINKS,SFT DRNK SNGL SRV BTL CARB (EX,National,GROCERY
685,1042438,COOKIES/CONES,SANDWICH COOKIES,National,GROCERY
1175,981521,BLEACH,LIQUID BLEACH,National,GROCERY
2002,962568,VEGETABLES - ALL OTHERS,CELERY,National,PRODUCE
10991,1004555,CAT LITTER,CONVENTIONAL LITTER,Private,GROCERY


In [54]:
recommend_for_user(212)

,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT
364,833025,MILK BY-PRODUCTS,SOUR CREAMS,Private,GROCERY
685,1042438,COOKIES/CONES,SANDWICH COOKIES,National,GROCERY
1633,911878,CORN,CORN WHITE,National,PRODUCE
5135,822785,COCOA MIXES,MALTED MLK/SYRUP/PWDRS (EGGNOG,Private,GROCERY
6365,873474,VALUE ADDED FRUIT,MELON HALVES/QUARTERS,National,PRODUCE


In [55]:
recommend_for_user(2305)

,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT
256,961554,CARROTS,CARROTS MINI PEELED,Private,PRODUCE
364,833025,MILK BY-PRODUCTS,SOUR CREAMS,Private,GROCERY
888,1068562,PROCESSED,PROCESSED OTHER,National,PRODUCE
7239,1104898,LUNCHMEAT,POULTRY,National,MEAT-PCKGD
37942,835595,TURKEY,GRND/PATTY - FROZEN,National,MEAT


In [56]:
recommend_for_user(480)

,PRODUCT_ID,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,DEPARTMENT
256,961554,CARROTS,CARROTS MINI PEELED,Private,PRODUCE
597,1037840,CIGARETTES,CIGARETTES,National,DRUG GM
685,1042438,COOKIES/CONES,SANDWICH COOKIES,National,GROCERY
9833,1013770,FRZN VEGETABLE/VEG DSH,FRZN BAGGED VEGETABLES - PLAIN,Private,GROCERY
27363,882190,DINNER SAUSAGE,SMOKED/COOKED,National,MEAT-PCKGD


In [57]:
# Question 2

In [58]:
merged_df.head()

,household_key,BASKET_ID,PRODUCT_ID,QUANTITY,SALES_VALUE,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,ProductText,AverageBasketSize,HouseholdSpending,PurchaseFrequency,ProductPopularity,InteractionScore
0,2375,26984851472,1004906,1,1.39,PRODUCE,Private,POTATOES,POTATOES RUSSET (BULK&BAG),POTATOES POTATOES RUSSET (BULK&BAG) Private PR...,10.333333,134.09,3,135,6.801079
1,2375,26984851472,1033142,1,0.82,PRODUCE,National,ONIONS,ONIONS SWEET (BULK&BAG),ONIONS ONIONS SWEET (BULK&BAG) National PRODUCE,10.333333,134.09,3,107,6.801079
2,2375,26984851472,1036325,1,0.99,PRODUCE,Private,VEGETABLES - ALL OTHERS,CELERY,VEGETABLES - ALL OTHERS CELERY Private PRODUCE,10.333333,134.09,3,86,6.801079
3,2375,26984851472,1082185,1,1.21,PRODUCE,National,TROPICAL FRUIT,BANANAS,TROPICAL FRUIT BANANAS National PRODUCE,10.333333,134.09,3,1141,6.801079
4,2375,26984851472,8160430,1,1.50,PRODUCE,Private,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,ORGANICS FRUIT & VEGETABLES ORGANIC CARROTS Pr...,10.333333,134.09,3,21,6.801079


In [59]:
transactions = (
    merged_df.groupby("BASKET_ID")["PRODUCT_ID"]
    .apply(list)
    .tolist()
)

In [60]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

encoded_data = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    encoded_data,
    columns=te.columns_
)     

# Apriori / FP-Growth

In [61]:
from algorithms import run_apriori, run_fp_growth, run_eclat, measure_time

min_support = 0.005

results = []

results.append(
    measure_time(
        "Apriori",
        lambda: run_apriori(basket, min_support)
    )
)

results.append(
    measure_time(
        "FP-Growth",
        lambda: run_fp_growth(basket, min_support)
    )
)

results_df = pd.DataFrame(results)

results_df

,Algorithm,Execution Time (seconds),Number of Frequent Itemsets
0,Apriori,1.8682,280
1,FP-Growth,1.4186,280


In [62]:
frequent_itemsets = run_fp_growth(basket, min_support)                                # pass arguments

frequent_itemsets = frequent_itemsets.sort_values(by = "support", ascending = False)  # Sort by highest support
frequent_itemsets.to_csv("frequent_itemsets.csv", index = False)                      # Save to CSV
print(frequent_itemsets.head(10))  

     support              itemsets
0   0.166521  frozenset({1082185})
30  0.080414  frozenset({1029743})
8   0.060274  frozenset({1106523})
81  0.047869   frozenset({995242})
5   0.045242   frozenset({981760})
13  0.039550   frozenset({904360})
18  0.038529  frozenset({1098066})
82  0.037361   frozenset({883404})
14  0.036486   frozenset({854852})
79  0.034880  frozenset({1127831})


In [63]:
from mlxtend.frequent_patterns import association_rules

frequent_itemsets = run_fp_growth(
    basket,
    min_support
)

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.1
)

rules = rules.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

print(rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
    "leverage",
    "conviction"
]].head(10))

             antecedents           consequents   support  confidence  \
3    frozenset({826249})  frozenset({1098066})  0.014886    0.445415   
4   frozenset({1098066})   frozenset({826249})  0.014886    0.386364   
51   frozenset({910032})  frozenset({1098066})  0.005108    0.318182   
52  frozenset({1098066})   frozenset({910032})  0.005108    0.132576   
22   frozenset({860776})   frozenset({995785})  0.006130    0.225806   
23   frozenset({995785})   frozenset({860776})  0.006130    0.213198   
49   frozenset({908531})   frozenset({995242})  0.005692    0.317073   
48   frozenset({995242})   frozenset({908531})  0.005692    0.118902   
15   frozenset({904360})   frozenset({854852})  0.007297    0.184502   
16   frozenset({854852})   frozenset({904360})  0.007297    0.200000   

         lift  leverage  conviction  
3   11.560540  0.013598    1.733676  
4   11.560540  0.013598    1.575166  
51   8.258264  0.004489    1.410158  
52   8.258264  0.004489    1.134331  
22   7.853938  0.

In [64]:
product_lookup = (
    merged_df[
        ['PRODUCT_ID', 'COMMODITY_DESC']
    ]
    .drop_duplicates()
)

product_dict = dict(
    zip(
        product_lookup['PRODUCT_ID'],
        product_lookup['COMMODITY_DESC']
    )
)

In [65]:
rules['antecedents'] = rules['antecedents'].apply(
    lambda x: [product_dict.get(i, i) for i in x]
)

rules['consequents'] = rules['consequents'].apply(
    lambda x: [product_dict.get(i, i) for i in x]
)

In [66]:
strong_rules = rules[
    (rules["confidence"] > 0.5) &
    (rules["lift"] > 1.5)
]

# Display strong rules
print(
    strong_rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift",
            "leverage",
            "conviction"
        ]
    ]
)

   antecedents       consequents   support  confidence      lift  leverage  \
56    [GRAPES]  [TROPICAL FRUIT]  0.005254    0.580645  3.486924  0.003747   
20    [GRAPES]  [TROPICAL FRUIT]  0.008319    0.537736  3.229243  0.005743   
28    [GRAPES]  [TROPICAL FRUIT]  0.007735    0.504762  3.031226  0.005183   

    conviction  
56    1.987528  
20    1.803037  
28    1.682987  


In [67]:
##question 3

In [69]:
#import 
from tkinter import *
from tkinter import ttk
from tkinter.scrolledtext import ScrolledText
from tkinter import messagebox
from tkinter.ttk import Progressbar 
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure


#window 
window = Tk()
window.title("Interactive Retail Analytics and Recommendation Dashboard")
window.geometry('1800x900')

lbl = Label(
    window,
    text="Designed for Adults Aged 65+ Using Accessible Visual Analytics",
    font=("Arial Bold", 20)
)

lbl.grid(column=0,row=0,columnspan=3,pady=20,sticky="ew")

product_lbl = Label(window,
    text="Select a Product:",
    font=("Arial", 16)
)

product_lbl.grid(column=0, row=1, pady=10)

dashboard_products = ["BANANAS","GRANOLA BARS","RAMEN NOODLES","SOFT DRINKS"]



#Combox

combo = ttk.Combobox( window,width=40,font=("Arial", 14)
)

combo['values'] = dashboard_products

combo.current(0)

combo.grid(column=1, row=1, pady=10)

output = ScrolledText(window,width=55,height=18,font=("Arial", 12),wrap=WORD)
output.grid(column=0,row=4,columnspan=3,padx=20,pady=20)


#Recemmemndatoions

def clicked():

    selected_product = combo.get()

    output.delete('1.0', END)

   

    if selected_product == "BANANAS":

        output.insert(
            END,
            "Original Product: BANANAS\n\n"
        )

        output.insert(
            END,
            "Recommended Products:\n"
        )

        output.insert(
            END,
            "- STRAWBERRIES (Similarity Score: 0.591)\n"
        )

        output.insert(
            END,
            "- TOMATOES (Similarity Score: 0.579)\n"
        )

        output.insert(
            END,
            "- ORANGES (Similarity Score: 0.529)\n"
        )



    elif selected_product == "RAMEN NOODLES":

        output.insert(
            END,
            "Original Product: RAMEN NOODLES\n\n"
        )

        output.insert(
            END,
            "Recommended Products:\n"
        )

        output.insert(
            END,
            "- JUICE DRINKS (Similarity Score: 0.769)\n"
        )

        output.insert(
            END,
            "- COLD CEREAL (Similarity Score: 0.764)\n"
        )

        output.insert(
            END,
            "- CANNED JUICES (Similarity Score: 0.723)\n"
        )

        output.insert(
            END,
            "- GARBAGE BAGS (Similarity Score: 0.682)\n"
        )


    elif selected_product == "SOFT DRINKS":

        output.insert(
            END,
            "Original Product: SOFT DRINKS\n\n"
        )

        output.insert(
            END,
            "Recommended Products:\n"
        )

        output.insert(
            END,
            "- CHIPS\n"
        )

        output.insert(
            END,
            "- FROZEN SNACKS\n"
        )

        output.insert(
            END,
            "- ICE CREAM\n"
        )

        output.insert(
            END,
            "- CANDY\n"
        )

    

    elif selected_product == "GRANOLA BARS":

        output.insert(
            END,
            "Original Product: GRANOLA BARS\n\n"
        )

        output.insert(
            END,
            "Recommended Products:\n"
        )

        output.insert(
            END,
            "- ADULT CEREAL (Similarity Score: 0.763)\n"
        )

        output.insert(
            END,
            "- FROZEN PIZZA (Similarity Score: 0.671)\n"
        )

        output.insert(
            END,
            "- LUNCHMEAT (Similarity Score: 0.594)\n"
        )

        output.insert(
            END,
            "- BATHROOM CLEANING PRODUCTS (Similarity Score: 0.517)\n"
        )



def create_charts():

  #Bar Charts 

    products = ["Bananas","Granola Bars","Soft Drinks","Ramen"]

    values = [120, 80, 100, 60]

    figure1 = Figure(figsize=(4,3), dpi=100)

    subplot1 = figure1.add_subplot(111)

    subplot1.bar(products,values,color="lightsteelblue")
    subplot1.set_title("Top Recommended Products")

    subplot1.set_ylabel("Recommendation Frequency")

    global bar1

    bar1 = FigureCanvasTkAgg(figure1,master=window)
    bar1.draw()
    bar1.get_tk_widget().grid(column=2,row=5,padx=20,pady=20)



#Pie Charts 

    departments = ["GROCERY","PRODUCE","DRINKS","SNACKS"]

    sizes = [45, 25, 15, 15]

    figure2 = Figure(figsize=(4,3), dpi=100)

    subplot2 = figure2.add_subplot(111)

    subplot2.pie(sizes,labels=departments,autopct='%1.1f%%')

    subplot2.set_title("Department Distribution")

    global pie2

    pie2 = FigureCanvasTkAgg(figure2,master=window)

    pie2.draw()

    pie2.get_tk_widget().grid(column=3,row=5,padx=20,pady=20)


#Line Charts 
    months = ["Jan","Feb","Mar","Apr","May","Jun"]
    sales = [20, 35, 30, 50, 45, 60]

    figure3 = Figure(figsize=(4,3),dpi=100)

    subplot3 = figure3.add_subplot(111)

    subplot3.plot(months,sales,marker='o',color='steelblue')

    subplot3.set_title("Monthly Product Purchases")

    subplot3.set_ylabel("Purchase Count")

    global line_chart

    line_chart = FigureCanvasTkAgg(figure3,master=window)

    line_chart.draw()

    line_chart.get_tk_widget().grid(column=1,row=5,padx=20, pady=20)


########
def clear_charts():

    bar1.get_tk_widget().destroy()

    pie2.get_tk_widget().destroy()

    line_chart.get_tk_widget().destroy()

#Botton

btn = Button(window,text="Generate Recommendations",font=("Arial Bold", 14),bg="palegreen",command=clicked)
btn.grid(column=0, row=3, padx=10)


#show chart bottton
chart_btn = Button(
    window, text="Create Charts", font=("Arial Bold", 14),bg="lightblue",command=create_charts)
chart_btn.grid(column=1, row=3, padx=10)



#clear Botton
clear_btn = Button( window,text="Clear Charts",font=("Arial Bold", 14),bg="salmon",command=clear_charts)

clear_btn.grid(column=2,row=3,padx=10)

#botton for exit 
exit_btn = Button(
    window,
    text="Exit Dashboard",font=("Arial Bold", 14), bg="lightgrey", command=window.destroy)

exit_btn.grid(column=3,row=3,padx=10)

window.mainloop()
